In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load data
data = pd.read_csv('/content/DataCoSupplyChainDataset.csv', encoding="ISO-8859-1")

In [ ]:
# Step 1: Data Preprocessing & Feature Engineering

# Convert order and shipping dates to datetime
data['order_date'] = pd.to_datetime(data['order date (DateOrders)'], errors='coerce')
data['shipping_date'] = pd.to_datetime(data['shipping date (DateOrders)'], errors='coerce')

# Calculate delay in days
data['shipping_delay'] = (data['shipping_date'] - data['order_date']).dt.days

In [ ]:
# Drop irrelevant columns as per initial exploration
remove_cat = ['Category Name', 'Customer City', 'Customer Country', 'Customer Email',
              'Customer Fname', 'Customer Lname', 'Customer Password', 'Customer State',
              'Customer Street', 'Department Name', 'Market', 'Order City', 'Order Country',
              'Order Region', 'Order State', 'Product Image', 'Product Name']

remove_cts = ['Category Id', 'Customer Id', 'Customer Zipcode', 'Department Id', 'Order Customer Id',
              'Order Id', 'Order Item Cardprod Id', 'Order Item Id', 'Order Zipcode', 'Product Card Id',
              'Product Category Id', 'Product Description', 'Product Status', 'Latitude', 'Longitude']

data.drop(columns=remove_cat + remove_cts, inplace=True)

In [ ]:
# Handle missing values if any
data.dropna(subset=['shipping_delay'], inplace=True)

# Encoding categorical data
data = pd.get_dummies(data, columns=['Type', 'Shipping Mode', 'Customer Segment', 'Order Status'], drop_first=True)

In [ ]:
# Step 2: Pricing Model Setup
# Assumptions for pricing
claim_payout = 100  # Claim payout amount
fixed_expense = 10  # Fixed expense per policy
target_profit_loading = 0.05  # Initial profit loading (may adjust if needed)

# Calculate the base premium
late_delivery_risk_prob = data['Late_delivery_risk'].mean()  # Assuming late_delivery_risk indicates probability
base_premium = late_delivery_risk_prob * claim_payout + fixed_expense

In [ ]:
# Step 3: Model Building and Evaluation

# Prepare features and target variable
X = data.drop(['Late_delivery_risk', 'order date (DateOrders)', 'shipping date (DateOrders)',
               'Delivery Status', 'order_date', 'shipping_date'], axis=1)
y = data['Late_delivery_risk']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Fit Logistic Regression Model
model = LogisticRegression()
model.fit(X_train, y_train)

# Model prediction
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 100.00%


In [ ]:
# Adjust Profit Loading based on Training Data Performance
claims_incurred = y_train.sum() * claim_payout  # Total claims paid in training data
total_premium_collected = len(y_train) * base_premium  # Total premium collected without profit loading

# Adjusting the profit loading if the current loading isn't sufficient
actual_profit_loading = (total_premium_collected - claims_incurred) / claims_incurred
if actual_profit_loading < target_profit_loading:
    adjusted_profit_loading = target_profit_loading - actual_profit_loading
    base_premium *= (1 + adjusted_profit_loading)

In [ ]:
# Step 4: Final Evaluation on Test Data

# Calculating profitability on test data
claims_incurred_test = y_test.sum() * claim_payout
total_premium_collected_test = len(y_test) * base_premium
actual_profit_test = total_premium_collected_test - claims_incurred_test

print(f"Test Set Profitability: {'Profitable' if actual_profit_test > 0 else 'Unprofitable'}")
print(f"Final Base Premium (with adjusted loading): ${base_premium:.2f}")

Test Set Profitability: Profitable
Final Base Premium (with adjusted loading): $64.83


Calculate Initial Profit Loading:
The code starts with a 5% profit premium (or profit loading) based on the "Equivalence Principle."

Using the claims paid out and the premiums collected in the training data, the code checks if a 5% profit premium achieves profitability.
If the initial 5% isn’t enough, the code increases the profit premium until break-even is achieved in the training data. This is done by calculating actual_profit_loading and adjusting it as needed.
Testing Data Evaluation:
After determining the adjusted premium, the code tests whether the premium also ensures profitability in the test dataset.


Findings: If the initial 5% profit premium was insufficient, note the final adjusted profit premium. For example, if the adjusted premium was, say, 7%, this indicates that a higher premium is necessary to account for the late delivery risk.

Observations:

Claim Frequency and Severity: If claims in the training data are high, a higher premium loading may be necessary to offset these payouts.
Overfitting Consideration: If the premium adjustment achieves profitability in the training data but not in the testing data, it may signal that the model needs further tuning to generalize well.
Customer Behavior and Risk Distribution: If the data shows high variability in delivery delays, that would explain the need for a higher premium loading than the standard.